In [7]:
!pip install shap

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
import shap
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
df = pd.read_csv('2_sql_processed_data.csv')

# 2. Preprocessing
le = LabelEncoder()
df['contract_type_encoded'] = le.fit_transform(df['contract_type'])
df['network_type_encoded'] = le.fit_transform(df['network_type'])

features = ['tenure_months', 'contract_type_encoded', 'network_type_encoded', 'monthly_bill',
            'avg_calls_per_month', 'days_since_recharge', 'frustration_index']
X = df[features]
y = df['churn_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Model Training (Random Forest)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=7)
rf_model.fit(X_train, y_train)

# Predict Probabilities for Segmentation later
df['churn_probability'] = rf_model.predict_proba(X)[:, 1]

# --- IMAGE 1: ROC AUC CURVE (Model Performance) ---
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='#34495e', lw=2, linestyle='--')
plt.title('Fig 1: Predictor Performance (ROC-AUC Curve)', fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('1_roc_curve.png', dpi=300)
plt.close()

# --- IMAGE 2: SHAP SUMMARY PLOT (AI Explainability) ---
# SHAP proves exactly WHICH features drive churn
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[..., 1], X_test, show=False)
plt.title('Fig 2: SHAP AI Explainability (What drives Churn?)', fontweight='bold')
plt.tight_layout()
plt.savefig('2_shap_summary.png', dpi=300, bbox_inches='tight')
plt.close()

# 4. CUSTOMER SEGMENTATION LOGIC (At Risk, Loyal, Dormant)
def assign_segment(row):
    if row['days_since_recharge'] > 60 and row['total_call_duration'] < 50:
        return 'Dormant'
    elif row['churn_probability'] > 0.60 or row['frustration_index'] > 8:
        return 'At Risk'
    else:
        return 'Loyal'

df['Customer_Segment'] = df.apply(assign_segment, axis=1)

# --- IMAGE 3: SEGMENTATION DISTRIBUTION ---
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='Customer_Segment', palette=['#3498db', '#e74c3c', '#95a5a6'])
plt.title('Fig 3: Total Customer Portfolio Segmentation', fontweight='bold')
plt.ylabel('Number of Customers')
plt.xlabel('Segment')
plt.tight_layout()
plt.savefig('3_customer_segments.png', dpi=300)
plt.close()

# --- IMAGE 4: THE 5G RETENTION IMPACT ---
plt.figure(figsize=(8, 6))
sns.barplot(x='network_type', y='churn_probability', data=df, palette='viridis')
plt.title('Fig 4: Churn Probability by Network Infrastructure', fontweight='bold')
plt.ylabel('Average AI Churn Probability')
plt.xlabel('Network Generation')
plt.axhline(df['churn_probability'].mean(), color='black', linestyle='--', label='Global Average')
plt.legend()
plt.tight_layout()
plt.savefig('4_network_impact.png', dpi=300)
plt.close()

# Export Final Data
df.to_csv('3_final_segmented_customers.csv', index=False)
print("SUCCESS: ML Complete. SHAP calculated. 4 PNGs generated and saved.")

SUCCESS: ML Complete. SHAP calculated. 4 PNGs generated and saved.
